In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND VALIDATION CONFIGURATION (PYTHON)
# ===================================================

from pyspark.sql import functions as F

CATALOG = "semiconplus_portfolio"
LANDING = f"{CATALOG}.simulation.simulated_retest_events_landing"
BRONZE = f"{CATALOG}.bronze.simulated_retest_events"
SILVER = f"{CATALOG}.silver.simulated_retest_events"
DIM_LOT = f"{CATALOG}.gold.dim_lot"
FACT_LOT = f"{CATALOG}.gold.fact_lot_performance"
FACT_PERIODIC = f"{CATALOG}.gold.fact_yield_periodic"
FACT_RETEST = f"{CATALOG}.gold.fact_retest_equipment"
EXPECTED_SEED = 20260818

print("Day 3 retest validation configuration loaded.")

In [0]:
# ===================================================
# BLOCK 2 — TABLE COUNTS AND RECONCILIATION (PYTHON)
# ===================================================

counts = {
    "landing": spark.table(LANDING).count(),
    "bronze": spark.table(BRONZE).count(),
    "silver": spark.table(SILVER).count(),
    "retest_fact": spark.table(FACT_RETEST).count(),
    "lot_fact": spark.table(FACT_LOT).count(),
    "periodic_fact": spark.table(FACT_PERIODIC).count(),
}

display(spark.createDataFrame(list(counts.items()), ["dataset", "row_count"]))
assert counts["landing"] == counts["bronze"] == counts["silver"]
assert counts["lot_fact"] == spark.table(DIM_LOT).count() == 18124
assert counts["periodic_fact"] == 17228

In [0]:
# ===================================================
# BLOCK 3 — SYNTHETIC DISCLOSURE AND UNIQUENESS (PYTHON)
# ===================================================

silver_df = spark.table(SILVER)

invalid_disclosure = silver_df.filter(
    (~F.col("simulated_record_flag"))
    | (F.col("simulation_seed") != EXPECTED_SEED)
    | (F.col("record_origin") != "DETERMINISTIC_PORTFOLIO_SIMULATION")
).count()

duplicate_events = (
    silver_df.groupBy("retest_event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_lots = (
    silver_df.groupBy("source_lot_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert invalid_disclosure == 0
assert duplicate_events == 0
assert duplicate_lots == 0
print("Synthetic disclosure, seed and uniqueness passed.")

In [0]:
# ===================================================
# BLOCK 4 — RETEST QUANTITY CONSTRAINTS (PYTHON)
# ===================================================

constraint_df = (
    silver_df.alias("retest")
    .join(
        spark.table(FACT_LOT).select(
            "source_lot_id",
            "input_quantity",
            "first_pass_good_quantity",
            "first_pass_fail_quantity",
        ).alias("fact"),
        "source_lot_id",
        "inner",
    )
)

invalid_constraints = constraint_df.filter(
    (F.col("retest_input_quantity") < 0)
    | (F.col("retest_input_quantity") > F.col("first_pass_fail_quantity"))
    | (F.col("retest_good_quantity") < 0)
    | (F.col("retest_good_quantity") > F.col("retest_input_quantity"))
    | (
        F.col("retest_input_quantity")
        != F.col("retest_good_quantity") + F.col("retest_fail_quantity")
    )
    | (
        F.col("first_pass_good_quantity") + F.col("retest_good_quantity")
        > F.col("input_quantity")
    )
).count()

assert invalid_constraints == 0, (
    f"Retest quantity constraint failures: {invalid_constraints}"
)
print("Retest quantity constraints passed.")

In [0]:
# ===================================================
# BLOCK 5 — FACT KEY AND GRAIN VALIDATION (PYTHON)
# ===================================================

retest_fact_df = spark.table(FACT_RETEST)

unresolved_keys = retest_fact_df.filter(
    F.col("date_key").isNull()
    | F.col("lot_key").isNull()
    | F.col("site_key").isNull()
    | F.col("product_group_key").isNull()
    | F.col("device_key").isNull()
    | F.col("equipment_key").isNull()
).count()

duplicate_grain = (
    retest_fact_df.groupBy(
        "retest_hour_utc",
        "lot_key",
        "equipment_key",
        "defect_code",
        "error_code",
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert unresolved_keys == 0
assert duplicate_grain == 0
print("Retest fact keys and grain passed.")

In [0]:
# ===================================================
# BLOCK 6 — SOURCE/FACT RETEST RECONCILIATION (PYTHON)
# ===================================================

def quantity_totals(table_name):
    return (
        spark.table(table_name)
        .agg(
            F.sum("retest_input_quantity").alias("retest_input_quantity"),
            F.sum("retest_good_quantity").alias("retest_good_quantity"),
            F.sum("retest_fail_quantity").alias("retest_fail_quantity"),
        )
        .first()
    )

silver_totals = quantity_totals(SILVER)
retest_fact_totals = quantity_totals(FACT_RETEST)
lot_fact_totals = quantity_totals(FACT_LOT)
periodic_totals = quantity_totals(FACT_PERIODIC)

totals_rows = []
for name, row in [
    ("SILVER", silver_totals),
    ("RETEST_FACT", retest_fact_totals),
    ("LOT_FACT", lot_fact_totals),
    ("PERIODIC_FACT", periodic_totals),
]:
    totals_rows.append((name, row[0], row[1], row[2]))

display(
    spark.createDataFrame(
        totals_rows,
        [
            "dataset",
            "retest_input_quantity",
            "retest_good_quantity",
            "retest_fail_quantity",
        ],
    )
)

assert len({tuple(row[1:]) for row in totals_rows}) == 1
assert silver_totals[0] == silver_totals[1] + silver_totals[2]


In [0]:
# ===================================================
# BLOCK 7 — FPY, FTY AND RECOVERY VALIDATION (PYTHON)
# ===================================================

lot_fact_df = spark.table(FACT_LOT)

invalid_yields = lot_fact_df.filter(
    (F.col("first_pass_yield") < 0)
    | (F.col("first_pass_yield") > 1)
    | (F.col("final_test_yield") < 0)
    | (F.col("final_test_yield") > 1)
    | (F.col("final_test_yield") < F.col("first_pass_yield"))
    | (F.col("final_good_quantity") > F.col("input_quantity"))
    | (
        F.abs(
            F.col("retest_recovery_contribution")
            - (F.col("final_test_yield") - F.col("first_pass_yield"))
        ) > F.lit(1e-12)
    )
    | (~F.col("retest_data_available_flag"))
).count()

assert invalid_yields == 0, f"Yield validation failures: {invalid_yields}"
print("FPY, FTY and recovery contribution passed.")

In [0]:
# ===================================================
# BLOCK 8 — DETERMINISTIC GENERATION SIGNATURE (PYTHON)
# ===================================================

signature = (
    silver_df
    .select(
        F.sha2(
            F.concat_ws(
                "|",
                "retest_event_id",
                "source_lot_id",
                "retest_input_quantity",
                "retest_good_quantity",
                "retest_fail_quantity",
                "simulation_seed",
            ),
            256,
        ).alias("row_signature")
    )
    .agg(
        F.count("*").alias("row_count"),
        F.min("row_signature").alias("minimum_row_signature"),
        F.max("row_signature").alias("maximum_row_signature"),
    )
)

display(signature)
print("Record these three values. They must be identical on the second run.")

In [0]:
# ===================================================
# BLOCK 9 — FINAL DAY 3 ACCEPTANCE RESULT (PYTHON)
# ===================================================

acceptance = {
    "synthetic_retest_rows": counts["silver"],
    "retest_fact_rows": counts["retest_fact"],
    "lot_fact_rows": counts["lot_fact"],
    "periodic_fact_rows": counts["periodic_fact"],
    "retest_input_quantity": int(silver_totals[0]),
    "retest_good_quantity": int(silver_totals[1]),
    "retest_fail_quantity": int(silver_totals[2]),
    "simulation_seed": EXPECTED_SEED,
    "simulated_record_flag": True,
    "status": "PASSED",
}

display(spark.createDataFrame([acceptance]))
print("DAY 3 MODEL ACCEPTANCE: PASSED")
print("Synthetic retest is ready for governed analytical reporting.")